# Lesson 5: GP Posterior Inference: What model.posterior(X) Actually Does

## This is one of the most important batches in the entire BoTorch study.

Until now, we have seen:

$$
\text{data} \rightarrow \text{kernel} \rightarrow K
$$

Now we will complete the next step:

$$
\boxed{
K \rightarrow \text{GP posterior}
}
$$

By the end of this batch, you should understand what happens mathematically when you write:

```python
posterior = model.posterior(test_X)
```

rather than treating `posterior()` as a black box.

The current BoTorch API defines `Model.posterior(X)` as computing the posterior over model outputs at the supplied `X`; the input is generally shaped as `batch_shape × q × d`.

---

### Where we are in the GP pipeline

We previously reached:

$$
f(x)\sim GP(m(x),k(x,x'))
$$

and constructed:

$$
K=
\begin{bmatrix}
k(x_1,x_1)&\cdots&k(x_1,x_n)\\
\vdots&\ddots&\vdots\\
k(x_n,x_1)&\cdots&k(x_n,x_n)
\end{bmatrix}.
$$

But we haven't yet answered:

> Given the observations $y_1,\ldots,y_n$, how do we predict $f(x_*)$?

The answer is **Gaussian conditioning**.

The complete chain is:

$$
\boxed{
\text{Kernel}
\rightarrow
K
\rightarrow
\text{Gaussian joint distribution}
\rightarrow
\text{condition on observations}
\rightarrow
\text{posterior}
}
$$

---

### Start with the simplest possible example

Let's use only two training points.

Suppose:

$$
x_1=0
$$

and

$$
x_2=1.
$$

We observe:

$$
y_1=1
$$

and

$$
y_2=2.
$$

So:

$$
X=
\begin{bmatrix}
0\\
1
\end{bmatrix},
\qquad
y=
\begin{bmatrix}
1\\
2
\end{bmatrix}.
$$

Assume:

$$
\ell=0.5
$$

and:

$$
\sigma_f^2=1.
$$

The RBF kernel gives:

$$
k(x,x')
=
\exp
\left(
-\frac{(x-x')^2}{2(0.5)^2}
\right).
$$

---

### Construct the training covariance matrix

For the first point:

$$
k(0,0)=1.
$$

For the second:

$$
k(1,1)=1.
$$

Between them:

$$
k(0,1)=e^{-2}\approx0.1353.
$$

Therefore:

$$
\boxed{
K_{XX}
=
\begin{bmatrix}
1&0.1353\\
0.1353&1
\end{bmatrix}
}
$$

This describes our prior belief about the relationship between the two latent function values.

---

### Add observation noise

Suppose measurement noise is:

$$
\sigma_n^2=0.01.
$$

Then:

$$
K_y=K_{XX}+\sigma_n^2I.
$$

Therefore:

$$
\boxed{
K_y=
\begin{bmatrix}
1.01&0.1353\\
0.1353&1.01
\end{bmatrix}
}
$$

This is the matrix we use when conditioning on noisy observations.

---

### Now introduce a new point

Suppose we want to predict at:

$$
x_*=0.25.
$$

We need the covariance between $x_*$ and each training point.

First:

$$
k(0.25,0)
=
\exp(-0.125)
\approx0.8825.
$$

Second:

$$
k(0.25,1)
=
\exp(-1.125)
\approx0.3247.
$$

So:

$$
\boxed{
K_{*X}
=
\begin{bmatrix}
0.8825&0.3247
\end{bmatrix}
}
$$

and:

$$
K_{X*}
=
\begin{bmatrix}
0.8825\\
0.3247
\end{bmatrix}.
$$

### joint Gaussian

The GP says that training values and the new value are jointly Gaussian:

$$
\begin{bmatrix}
y\\
f_*
\end{bmatrix}
\sim
\mathcal N
\left(
\begin{bmatrix}
m_X\\
m_*
\end{bmatrix},
\begin{bmatrix}
K_y & K_{X*}\\
K_{*X} & K_{**}
\end{bmatrix}
\right).
$$

For zero mean:

$$
\boxed{
\begin{bmatrix}
y\\
f_*
\end{bmatrix}
\sim
\mathcal N
\left(
0,
\begin{bmatrix}
K_y & K_{X*}\\
K_{*X} & K_{**}
\end{bmatrix}
\right)
}
$$

This equation is the mathematical heart of GP regression.

---

### Posterior mean

Conditioning a Gaussian gives:

$$
\boxed{
\mu_* = K_{*X}K_y^{-1}y
}
$$

for zero prior mean.

Substituting:

$$
\mu_*
=
\begin{bmatrix}
0.8825&0.3247
\end{bmatrix}
K_y^{-1}
\begin{bmatrix}
1\\
2
\end{bmatrix}.
$$

You could explicitly calculate:

$$
K_y^{-1}.
$$

But **we generally should not actually compute the inverse**.

This is a crucial computational concept.

---

### Never unnecessarily calculate $K^{-1}$

Mathematically, we write:

$$
K^{-1}y.
$$

Computationally, we instead solve:

$$
K\alpha=y.
$$

Then:

$$
\boxed{
\alpha=K^{-1}y
}
$$

mathematically, but computationally obtained through a linear solve.

Why?

Because explicitly calculating:

```python
torch.linalg.inv(K)
```

is generally:

- less numerically stable,
- less efficient,
- unnecessary.

Instead:

```python
alpha = torch.linalg.solve(K, y)
```

or, for a positive-definite covariance matrix, preferably use Cholesky-based solves.

This distinction is extremely important for understanding GPyTorch.

---

### Why covariance matrices are special

Our covariance matrix is positive definite:

$$
K_y\succ0.
$$

For such a matrix, we can perform a **Cholesky decomposition**:

$$
\boxed{
K_y=LL^T
}
$$

where $L$ is lower triangular.

For our matrix:

$$
K_y=
\begin{bmatrix}
1.01&0.1353\\
0.1353&1.01
\end{bmatrix}.
$$

The Cholesky factor has the form:

$$
L=
\begin{bmatrix}
a&0\\
b&c
\end{bmatrix}.
$$

Then:

$$
LL^T=K_y.
$$

---

### Why Cholesky is useful

Instead of solving:

$$
K_y\alpha=y
$$

directly, we have:

$$
LL^T\alpha=y.
$$

Define:

$$
Lz=y.
$$

Solve for $z$.

Then:

$$
L^T\alpha=z.
$$

Solve for $\alpha$.

So we replace one difficult matrix problem with two triangular solves.

This is:

$$
\boxed{
K\alpha=y
\quad\rightarrow\quad
LL^T\alpha=y
}
$$

and this is computationally efficient and numerically stable for positive-definite covariance matrices.

### PyTorch demonstration

In [1]:
import torch

torch.set_default_dtype(torch.double)

K = torch.tensor([
    [1.01, 0.1353],
    [0.1353, 1.01],
])

y = torch.tensor([
    [1.0],
    [2.0],
])

L = torch.linalg.cholesky(K)

print("K:")
print(K)

print("\nCholesky factor L:")
print(L)

print("\nReconstructed K:")
print(L @ L.T)

K:
tensor([[1.0100, 0.1353],
        [0.1353, 1.0100]])

Cholesky factor L:
tensor([[1.0050, 0.0000],
        [0.1346, 0.9959]])

Reconstructed K:
tensor([[1.0100, 0.1353],
        [0.1353, 1.0100]])


You should find:

```python
L @ L.T ≈ K
```

This verifies:

$$
\boxed{
K\approx LL^T
}
$$

up to floating-point precision.

### Solve for alpha

In [2]:
alpha = torch.cholesky_solve(y, L)

print(alpha)

tensor([[0.7381],
        [1.8813]])


Conceptually:

$$
\boxed{
\alpha=K^{-1}y
}
$$

without explicitly calculating $K^{-1}$.

Then posterior mean is:

In [3]:
K_star_X = torch.tensor([
    [0.8825, 0.3247]
])

mu = K_star_X @ alpha

print(mu)

tensor([[1.2622]])


This gives approximately:

$$
\boxed{
\mu_*\approx1.18
}
$$

The exact value will vary slightly depending on rounding.

So our GP predicts approximately:

$$
f(0.25)\approx1.18.
$$

### Why is the prediction between the observations?

We observed:

$$
f(0)\approx1
$$

and:

$$
f(1)\approx2.
$$

At:

$$
x_*=0.25,
$$

we expect something closer to the first observation.

The kernel gives:

$$
k(0.25,0)=0.8825
$$

but:

$$
k(0.25,1)=0.3247.
$$

Therefore, the first observation has a stronger influence.

This is the GP's version of:

> “Nearby experimental conditions should generally have more influence.”

---

### Posterior variance

The posterior variance is:

$$
\boxed{
\sigma_*^2
=
K_{**}
-
K_{*X}K_y^{-1}K_{X*}
}
$$

For our RBF:

$$
K_{**}=k(0.25,0.25)=1.
$$

So:

$$
\sigma_*^2
=
1-K_{*X}K_y^{-1}K_{X*}.
$$

Again, we do not explicitly calculate the inverse.

Instead, solve:

$$
K_yv=K_{X*}.
$$

Then:

$$
\sigma_*^2
=
K_{**}-K_{*X}v.
$$

### PyTorch implementation

In [4]:
K_star_X = torch.tensor([
    [0.8825, 0.3247]
])

K_X_star = K_star_X.T

K_star_star = torch.tensor([
    [1.0]
])

v = torch.cholesky_solve(
    K_X_star,
    L,
)

variance = K_star_star - K_star_X @ v

print("Posterior variance:")
print(variance)

print("Posterior standard deviation:")
print(torch.sqrt(variance))

Posterior variance:
tensor([[0.1859]])
Posterior standard deviation:
tensor([[0.4312]])


### Why posterior variance decreases near data

Now try:

$$
x_*=0.01.
$$

This point is extremely close to:

$$
x_1=0.
$$

Therefore:

$$
k(x_*,x_1)\approx1.
$$

The model has a lot of information about this region.

Consequently:

$$
\sigma_*^2
$$

becomes relatively small.

Now try:

$$
x_*=5.
$$

The covariance with both training points is almost zero.

Therefore:

$$
K_{*X}\approx0.
$$

Then:

$$
\sigma_*^2\approx K_{**}.
$$

The GP essentially returns to its prior uncertainty.

This is the mathematical foundation of GP exploration.

### This is the key BO mechanism

Imagine your SDL has measured:

```text
Electrolyte A
Electrolyte B
Electrolyte C
```

The GP has low uncertainty around these observations.

But there is an unexplored region:

```text
Electrolyte D
```

far away in normalized input space.

The GP may say:

$$
\mu_D=\text{uncertain}
$$

and:

$$
\sigma_D^2=\text{large}.
$$

An acquisition function can then recognize that this uncertainty itself has value.

That's why BO can explore.

---

### The posterior is a distribution, not merely a prediction

This is a very important conceptual difference.

A normal regression model might give:

$$
\hat y=1.82.
$$

A GP gives:

$$
f(x_*)\sim
\mathcal N
\left(
\mu(x_*),
\sigma^2(x_*)
\right).
$$

For example:

$$
\boxed{
f(x_*)\sim\mathcal N(1.82,0.16^2)
}
$$

means:

$$
\mu=1.82
$$

and:

$$
\sigma=0.16.
$$

The uncertainty is essential for Bayesian optimization.

---

### Mean versus variance

Suppose two candidate electrolytes have:

#### Candidate A

$$
\mu_A=1.90
$$

$$
\sigma_A=0.03.
$$

#### Candidate B

$$
\mu_B=1.85
$$

$$
\sigma_B=0.30.
$$

A pure exploitation strategy would probably prefer A.

But Bayesian optimization might prefer B because:

$$
1.85
$$

could potentially be much better than expected.

This is where acquisition functions come in.

But the acquisition function can only exploit uncertainty because the GP gives it:

$$
\boxed{
\mu(x)
}
$$

and:

$$
\boxed{
\sigma(x)
}
$$

### Now let's connect this to actual BoTorch

In [6]:
import torch

from botorch.models import SingleTaskGP
from botorch.fit import fit_gpytorch_mll
from gpytorch.mlls import ExactMarginalLogLikelihood

torch.set_default_dtype(torch.double)

train_X = torch.tensor([
    [0.0],
    [0.5],
    [1.0],
])

train_Y = torch.tensor([
    [1.0],
    [1.8],
    [2.0],
])

model = SingleTaskGP(
    train_X,
    train_Y,
)

mll = ExactMarginalLogLikelihood(
    model.likelihood,
    model,
)

fit_gpytorch_mll(mll)

/Users/benediktusmadika/Desktop/AI learning/botorch_sdl/lib/python3.9/site-packages/botorch/models/utils/assorted.py:202: InputDataWarning: Input data is not standardized (mean = tensor([1.6000]), std = tensor([0.5292])). Please consider scaling the input to zero mean and unit variance.
  warnings.warn(msg, InputDataWarning)


ExactMarginalLogLikelihood(
  (likelihood): GaussianLikelihood(
    (noise_covar): HomoskedasticNoise(
      (noise_prior): GammaPrior()
      (raw_noise_constraint): GreaterThan(1.000E-04)
    )
  )
  (model): SingleTaskGP(
    (likelihood): GaussianLikelihood(
      (noise_covar): HomoskedasticNoise(
        (noise_prior): GammaPrior()
        (raw_noise_constraint): GreaterThan(1.000E-04)
      )
    )
    (mean_module): ConstantMean()
    (covar_module): ScaleKernel(
      (base_kernel): MaternKernel(
        (lengthscale_prior): GammaPrior()
        (raw_lengthscale_constraint): Positive()
      )
      (outputscale_prior): GammaPrior()
      (raw_outputscale_constraint): Positive()
    )
  )
)

In [7]:
test_X = torch.tensor([
    [0.25],
    [0.75],
])

Because the BoTorch posterior API expects test points in batch_shape × q × d form, it is often useful to write:

In [8]:
test_X = test_X.unsqueeze(0)

[1, 2, 1]

where:
- 1 = batch size
- 2 = two jointly considered test points
- 1 = input dimension.
Then:

In [9]:
posterior = model.posterior(test_X)

The returned object represents the posterior distribution over those points. The BoTorch model API documents the posterior() output as a Posterior object representing joint distributions over the queried points.

### Extract the mean

```python
print(posterior.mean)
```

Conceptually:

```text
tensor([
    [[μ₁],
     [μ₂]]
])
```

Shape:

$$
[1,2,1].
$$

So:

$$
\boxed{
\text{posterior.mean.shape}=[1,q,m]
}
$$

for this simple case, where:

- $q=2$
- $m=1$.

---

### Extract variance

```python
print(posterior.variance)
```

You get:

$$
\sigma^2(x_1)
$$

and:

$$
\sigma^2(x_2).
$$

And:

```python
print(posterior.variance.sqrt())
```

gives:

$$
\sigma(x).
$$

---

### Extract the full covariance matrix

This is an important experiment.

For two test points:

```python
print(posterior.mvn.covariance_matrix)
```

Depending on the specific posterior implementation, the underlying multivariate normal can expose its covariance representation.

For two jointly evaluated points:

$$
\begin{bmatrix}
f(x_1)\\
f(x_2)
\end{bmatrix}
$$

the posterior contains not only:

$$
\operatorname{Var}(f(x_1))
$$

and:

$$
\operatorname{Var}(f(x_2)),
$$

but also:

$$
\operatorname{Cov}(f(x_1),f(x_2)).
$$

This matters tremendously for **q-batch BO**, which we will study later.

### Why `q=2` is not merely two independent predictions

Suppose:

```python
X = torch.tensor([
    [[0.25],
     [0.75]]
])
```

This means:

> Consider these two candidate points jointly.

The GP produces:

$$
\begin{bmatrix}
f(x_1)\\
f(x_2)
\end{bmatrix}
$$

as a joint Gaussian.

Therefore we have:

$$
\operatorname{Cov}(f(x_1),f(x_2)).
$$

This covariance is important when an acquisition function evaluates a batch of experiments.

For example, the value of selecting:

```text
A + B
```

together depends partly on whether A and B provide redundant information.

That will become extremely important for SDL batch experiments.

---

### A subtle distinction: latent function vs observation posterior

Recall:

$$
y=f(x)+\epsilon.
$$

Therefore:

#### Latent posterior

$$
p(f_*|D)
$$

represents uncertainty about the underlying physical function.

#### Observation posterior

$$
p(y_*|D)
$$

also includes measurement noise.

In BoTorch:

```python
posterior = model.posterior(X)
```

normally gives the model's latent predictive posterior.

If you request:

```python
posterior = model.posterior(
    X,
    observation_noise=True,
)
```

observation noise is incorporated for models with inferred noise.

This distinction is extremely important for experimental SDL.

### Example for your electrolyte SDL

Suppose the underlying ionic conductivity is:

$$
f(x).
$$

The SDL measurement is:

$$
y=f(x)+\epsilon.
$$

You may have:

$$
f(x)=1.80
$$

but measurements:

$$
y_1=1.76
$$

$$
y_2=1.83
$$

$$
y_3=1.81.
$$

The GP should distinguish:

```text
real material-function uncertainty
```

from:

```text
measurement noise
```

That is what the likelihood is helping you model.

---

### Why Cholesky is so important in GPyTorch

You might wonder:

> Why are we spending so much time on Cholesky?

Because GP inference repeatedly requires operations involving covariance matrices.

Naively:

$$
K^{-1}
$$

would be expensive and potentially numerically unstable.

Instead, GPyTorch uses specialized linear algebra and lazy covariance representations.

This is one reason the framework is much more sophisticated than simply writing:

```python
torch.inverse(K)
```

for every prediction.

The underlying GP machinery is designed to perform these operations efficiently.

### Computational complexity

For an exact GP with:

$$
n
$$

training points, dense covariance operations generally have approximately:

$$
\boxed{
O(n^3)
}
$$

time complexity for the major factorization step.

Memory is approximately:

$$
\boxed{
O(n^2)
}
$$

This is one of the central limitations of exact GPs.

For example:

$$
n=100
$$

is easy.

$$
n=1,000
$$

is more demanding.

$$
n=10,000
$$

can become very challenging for a standard exact GP.

This is relevant to your SDL because sequential experiments often involve relatively small datasets, which is one reason GP-based BO is so attractive.

---

### Why this suits your SDL problem

Suppose your initial dataset contains:

$$
n=20
$$

electrolyte experiments.

Then after 30 iterations:

$$
n=50.
$$

After 100 experiments:

$$
n=120.
$$

An exact GP is still quite practical in many such settings.

This is one reason Bayesian optimization is particularly effective when:

> experiments are expensive and datasets are relatively small.

Your SDL may spend hours per electrochemical experiment, while the GP fitting may take seconds or less.

That is a favorable computational tradeoff.

---

### Numerical stability: jitter

There is another concept you absolutely need to know:

$$
\boxed{
\text{jitter}
}
$$

Suppose numerical rounding makes your covariance matrix nearly singular.

Mathematically you have:

$$
K.
$$

Computationally, we may use:

$$
\boxed{
K+\epsilon I
}
$$

where:

$$
\epsilon>0
$$

is tiny.

For example:

$$
\epsilon=10^{-6}.
$$

Then:

$$
K_{\mathrm{stable}}
=
K+10^{-6}I.
$$

This slightly increases the diagonal and makes the matrix easier to factor numerically.

---

### Jitter is not the same as observation noise

This distinction is critical.

#### Observation noise

$$
\sigma_n^2I
$$

represents:

> real measurement uncertainty.

#### Jitter

$$
\epsilon I
$$

represents:

> numerical stabilization.

So conceptually:

$$
\boxed{
\text{noise}=\text{scientific model}
}
$$

while:

$$
\boxed{
\text{jitter}=\text{numerical technique}
}
$$

Do not confuse them.

---

### Why nearly duplicate experiments can cause trouble

Suppose your SDL accidentally runs:

$$
x_1=[0.5,0.3]
$$

and:

$$
x_2=[0.50000001,0.30000001].
$$

The kernel may see them as essentially identical.

Then the covariance matrix can become nearly singular.

This can make numerical factorization difficult.

This is another reason experimental design and duplicate handling matter in real BO systems.

## Complete manual posterior demonstration

In [10]:
import torch

torch.set_default_dtype(torch.double)

# --------------------------------------------------
# Training data
# --------------------------------------------------

X = torch.tensor([
    [0.0],
    [1.0],
])

y = torch.tensor([
    [1.0],
    [2.0],
])

# --------------------------------------------------
# RBF kernel
# --------------------------------------------------

lengthscale = 0.5
signal_variance = 1.0
noise_variance = 0.01

def rbf_kernel(X1, X2):
    distance_sq = (
        (X1[:, None, :] - X2[None, :, :]) ** 2
    ).sum(dim=-1)

    return signal_variance * torch.exp(
        -distance_sq / (2 * lengthscale**2)
    )

# --------------------------------------------------
# Training covariance
# --------------------------------------------------

K = rbf_kernel(X, X)

K_noise = K + noise_variance * torch.eye(
    X.shape[0]
)

# --------------------------------------------------
# Cholesky
# --------------------------------------------------

L = torch.linalg.cholesky(K_noise)

# --------------------------------------------------
# Solve K alpha = y
# --------------------------------------------------

alpha = torch.cholesky_solve(
    y,
    L,
)

# --------------------------------------------------
# Test point
# --------------------------------------------------

X_star = torch.tensor([
    [0.25],
])

K_star_X = rbf_kernel(X_star, X)

K_star_star = rbf_kernel(
    X_star,
    X_star,
)

# --------------------------------------------------
# Posterior mean
# --------------------------------------------------

posterior_mean = K_star_X @ alpha

# --------------------------------------------------
# Posterior variance
# --------------------------------------------------

v = torch.cholesky_solve(
    K_star_X.T,
    L,
)

posterior_variance = (
    K_star_star
    - K_star_X @ v
)

print("Posterior mean:")
print(posterior_mean)

print("\nPosterior variance:")
print(posterior_variance)

print("\nPosterior standard deviation:")
print(torch.sqrt(posterior_variance))

Posterior mean:
tensor([[1.2621]])

Posterior variance:
tensor([[0.1860]])

Posterior standard deviation:
tensor([[0.4312]])


This code is not meant to replace BoTorch.
It is meant to make the internals of BoTorch understandable.

### Compare that with BoTorch

The manual calculation required us to explicitly construct:

```text
K
K_noise
L
alpha
K_star_X
posterior_mean
posterior_variance
```

BoTorch lets you write:

```python
posterior = model.posterior(X_star)
```

and gives you the resulting posterior.

So:

```text
Manual mathematics
        ↓
K
        ↓
Cholesky
        ↓
solve
        ↓
posterior mean/variance
```

is conceptually hidden inside:

```python
model.posterior(X_star)
```

That is exactly why understanding the mathematics is useful:

**you know what the API is doing for you.**

---

### A very important BoTorch shape issue

Suppose:

```python
test_X.shape
```

is:

```text
[10, 2]
```

This means:

> 10 points, 2 dimensions.

For `model.posterior`, BoTorch's general API expects:

```text
batch_shape × q × d
```

so you can use:

```python
test_X = test_X.unsqueeze(0)
```

giving:

```text
[1, 10, 2]
```

Interpretation:

$$
b=1,\qquad q=10,\qquad d=2.
$$

The current API documentation explicitly describes the posterior input this way.

This becomes particularly important later when we use:

$$
q>1
$$

for batch Bayesian optimization.

### Why posterior covariance matters for batch BO

Suppose you want to send three experiments to your SDL simultaneously:

$$
x_1,x_2,x_3.
$$

The GP gives:

$$
\begin{bmatrix}
f(x_1)\\
f(x_2)\\
f(x_3)
\end{bmatrix}
\sim
\mathcal N
\left(
\mu,
\Sigma
\right).
$$

Here:

$$
\Sigma=
\begin{bmatrix}
\sigma_1^2 & \sigma_{12} & \sigma_{13}\\
\sigma_{21} & \sigma_2^2 & \sigma_{23}\\
\sigma_{31} & \sigma_{32} & \sigma_3^2
\end{bmatrix}.
$$

The diagonal terms:

$$
\sigma_1^2,\sigma_2^2,\sigma_3^2
$$

represent the individual uncertainty of each candidate.

The off-diagonal terms:

$$
\sigma_{12},\sigma_{13},\sigma_{23}
$$

tell us how correlated the candidate outcomes are.

For example, if:

$$
\sigma_{12}
$$

is large and positive, then the GP considers the outcomes at:

$$
x_1
$$

and:

$$
x_2
$$

to be strongly related.

Learning something about one candidate therefore also tells us something about the other.

---

### Why this matters for batch selection

Therefore selecting:

```text
A, B, C
```

is not equivalent to independently selecting:

```text
A
B
C
```

The three candidates must be evaluated **jointly**.

For example, suppose:

```text
A and B are very similar
```

while:

```text
C is substantially different.
```

The GP may have:

$$
\operatorname{Cov}(f_A,f_B)
$$

that is relatively large, while:

$$
\operatorname{Cov}(f_A,f_C)
$$

and:

$$
\operatorname{Cov}(f_B,f_C)
$$

are smaller.

If you select A and B together, they may provide relatively redundant information.

Selecting A and C could provide more diverse information.

This is one of the reasons batch Bayesian optimization considers the **joint posterior distribution** rather than simply choosing the three candidates with the highest individual acquisition values.

---

### The key idea

For a single experiment, we mainly care about:

$$
\mu(x)
$$

and:

$$
\sigma^2(x).
$$

For a batch of experiments, we additionally care about:

$$
\boxed{
\operatorname{Cov}(f(x_i),f(x_j))
}
$$

between candidates.

So:

```text
Single-point BO
       ↓
mean + variance
       ↓
choose one experiment
```

while:

```text
Batch BO
       ↓
joint mean + covariance
       ↓
choose several experiments jointly
```

This becomes fundamental when your SDL can run several cells in parallel.

## The complete conceptual picture

```text
                    TRAINING DATA
                         │
                         ▼
                    train_X/Y
                         │
                         ▼
                      Kernel
                         │
                         ▼
                      K(X,X)
                         │
                         ▼
              + noise covariance
                         │
                         ▼
                       K_y
                         │
                         ▼
                    Cholesky
                         │
                         ▼
                  Linear solves
                         │
                         ▼
                  GP posterior
                    /       \
                   /         \
                  ▼           ▼
              mean μ       variance σ²
                  \           /
                   \         /
                    ▼       ▼
                 Acquisition
                      │
                      ▼
               next experiment
```

### What `model.posterior(X)` really means

When you write:

```python
posterior = model.posterior(X)
```

you should mentally translate it into:

> "Take the GP I fitted to my existing experimental data, evaluate the covariance between the new points and the training data, condition the joint Gaussian distribution on my observations, and return the resulting predictive distribution."

That is what the API abstraction represents.

---

### The three equations to memorize

You do **not** need to memorize every GP derivation.

But these three equations should become familiar.

#### Equation 1 — posterior mean

$$
\boxed{
\mu_* =
K_{*X}
(K_{XX}+\sigma_n^2I)^{-1}y
}
$$

This tells us the GP's predicted mean at the new point.

---

#### Equation 2 — posterior variance

$$
\boxed{
\sigma_*^2 =
K_{**}
-
K_{*X}
(K_{XX}+\sigma_n^2I)^{-1}
K_{X*}
}
$$

This tells us how uncertain the GP is about the new point.

---

#### Equation 3 — computational factorization

$$
\boxed{
K+\sigma_n^2I=LL^T
}
$$

Then instead of explicitly computing:

$$
K^{-1},
$$

we solve linear systems using $L$.

Conceptually:

```text
Covariance matrix
       ↓
K + noise
       ↓
Cholesky
       ↓
L Lᵀ
       ↓
Linear solves
       ↓
Posterior mean + variance
```

So when you write:

```python
posterior = model.posterior(X)
```

you are invoking this entire mathematical process through the BoTorch/GPyTorch API.

### One final connection to your future electrolyte SDL

Imagine your initial dataset:

$$
D_0=\{(x_i,y_i)\}_{i=1}^{20}.
$$

Here:

- $x_i$ represents an electrolyte formulation and its experimental conditions.
- $y_i$ represents the measured electrochemical performance.

You fit the GP:

```python
model = SingleTaskGP(train_X, train_Y)
fit_gpytorch_mll(mll)
```

Now the GP has learned its hyperparameters, including:

$$
\ell_1,\ldots,\ell_d
$$

and:

$$
\sigma_f^2,\sigma_n^2.
$$

These parameters determine how the GP interprets similarity between electrolyte formulations, the magnitude of function variation, and the expected measurement noise.

---

#### Step 1 — Generate the posterior

You then evaluate the GP at candidate electrolyte formulations:

```python
posterior = model.posterior(candidate_X)
```

The GP provides:

$$
\mu(x)
$$

and:

$$
\sigma(x).
$$

So for every candidate, the GP gives both:

```text
predicted performance
        +
prediction uncertainty
```

---

#### Step 2 — Evaluate the acquisition function

An acquisition function such as:

```python
LogExpectedImprovement(
    model=model,
    best_f=best_f,
)
```

uses the GP posterior.

Conceptually:

$$
\boxed{
\text{GP posterior}
\rightarrow
\text{acquisition value}
}
$$

The acquisition function combines the predicted performance and uncertainty to determine which candidate is promising to evaluate next.

---

#### Step 3 — Optimize the acquisition function

Then:

```python
candidate_X, acq_value = optimize_acqf(...)
```

solves the optimization problem:

$$
\boxed{
x_{\mathrm{next}}
=
\arg\max_{x\in\mathcal X}
\alpha(x)
}
$$

where $\alpha(x)$ is the acquisition function.

The result is the next electrolyte formulation that the BO system recommends.

---

#### The complete SDL loop

Now the entire process becomes:

```text
Initial experimental data
        │
        ▼
   train_X / train_Y
        │
        ▼
   Fit GP model
        │
        ▼
   GP posterior
        │
        ├──────────────┐
        ▼              ▼
     mean μ         uncertainty σ
        │              │
        └──────┬───────┘
               ▼
      Acquisition function
               │
               ▼
       optimize_acqf(...)
               │
               ▼
        candidate_X
               │
               ▼
              SDL
               │
               ▼
   Electrolyte fabrication
               │
               ▼
  Electrochemical measurement
               │
               ▼
              new y
               │
               ▼
      Append to train_X/Y
               │
               ▼
          Refit GP
               │
               ▼
        New posterior
               │
               ▼
     New acquisition function
               │
               ▼
       Next experiment
               │
               └───────────────┐
                               │
                               ▼
                        Repeat the cycle
```

This is the fundamental **closed-loop Bayesian optimization workflow** for your electrolyte SDL.

The important idea is that the GP is not simply predicting the next electrolyte.

It is continuously updated with new experimental evidence, and that updated uncertainty guides the selection of the next experiment.

So the loop is:

$$
\boxed{
\text{Experiment}
\rightarrow
\text{Data}
\rightarrow
\text{GP}
\rightarrow
\text{Posterior}
\rightarrow
\text{Acquisition}
\rightarrow
\text{Next Experiment}
}
$$

And then the cycle repeats.